In [1]:
import os
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import json

## Load Data

In [2]:
# filtered_data_path = "/data2/czbenchmarks/replogle2022/K562_essential_raw_singlecell_01.h5ad"
filtered_data_path = f"{os.environ['HOME']}/.cz-benchmarks/datasets/replogle_k562_essential_perturbpredict_de_results_control_cells_v1.h5ad"

adata_filtered = ad.read_h5ad(filtered_data_path, backed=None)
adata_filtered.obs = adata_filtered.obs.rename(columns={'gene_id':'condition', 'gene':'condition_name'})
adata_filtered.var = adata_filtered.var.rename(columns={'gene_name':'gene'})
adata_filtered.obs['condition'] = adata_filtered.obs['condition'].astype(str)

control_cells_ids_cr4_orig = adata_filtered.uns["control_cells_ids"]
control_cells_ids_cr4_orig.pop("non-targeting", None)

array([], dtype=float64)

In [3]:
cr4_data_path = (
    "/data2/czbenchmarks/replogle2022/raw_h5ad_from_cr4/K562_essential_mtx.h5ad"
)
adata_cr4 = ad.read_h5ad(cr4_data_path, backed=None)
adata_cr4.var.index.name = "gene_id"
adata_cr4.var.rename(columns={"gene_name": "gene"}, inplace=True)
adata_cr4.obs.rename(columns={"gene_id": "condition"}, inplace=True)

In [4]:
with open("/data2/czbenchmarks/control_cells_ids_replogle_k562_essential_perturbpredict.json", "r") as f:
    control_cells_ids_filtered = json.load(f) # filtered data

with open("/data2/czbenchmarks/control_cells_ids_replogle_k562_essential_perturbpredict_validate.json", "r") as f:
    control_cells_ids_cr4_validate = json.load(f) # cr4 data

## QC of Conditions

In [5]:
len(control_cells_ids_cr4_orig.keys()), len(control_cells_ids_filtered.keys()), len(control_cells_ids_cr4_validate.keys())

(2057, 2057, 2057)

In [6]:
set(control_cells_ids_filtered.keys()) == set(control_cells_ids_cr4_orig.keys()) == set(control_cells_ids_cr4_validate.keys())

True

In [7]:
set(control_cells_ids_filtered.keys()).issubset(set(control_cells_ids_cr4_orig.keys()))

True

In [8]:
set(control_cells_ids_cr4_validate.keys()).issubset(set(control_cells_ids_cr4_orig.keys()))

True

## QC of Matches

In [153]:
# conditions = list(control_cells_ids_filtered.keys())
conditions = list(control_cells_ids_cr4_orig.keys())

# Old and new implementation fairly similar when both run on same dataset (cr4)
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_orig[condition]), len(control_cells_ids_cr4_validate[condition]))
    assert len(control_cells_ids_cr4_orig[condition]) == len(control_cells_ids_cr4_validate[condition])
    
    matches = 0
    for pos, (val1, val2) in enumerate(zip(control_cells_ids_cr4_orig[condition], control_cells_ids_cr4_validate[condition].values())):
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {len(control_cells_ids_cr4_orig[condition]) - matches}")

ENSG00000001497 83 83
     matches 83, differences 0
ENSG00000003509 18 18
     matches 18, differences 0
ENSG00000004779 160 160
     matches 159, differences 1
ENSG00000004897 366 366
     matches 364, differences 2
ENSG00000005007 131 131
     matches 131, differences 0
ENSG00000005100 140 140
     matches 140, differences 0
ENSG00000005175 18 18
     matches 18, differences 0
ENSG00000005194 151 151
     matches 151, differences 0
ENSG00000005448 208 208
     matches 206, differences 2
ENSG00000006634 131 131
     matches 130, differences 1
ENSG00000006695 94 94
     matches 94, differences 0
ENSG00000006712 238 238
     matches 235, differences 3
ENSG00000006715 332 332
     matches 331, differences 1
ENSG00000006744 85 85
     matches 85, differences 0
ENSG00000007168 126 126
     matches 123, differences 3
ENSG00000007866 267 267
     matches 265, differences 2
ENSG00000007923 46 46
     matches 46, differences 0
ENSG00000008018 143 143
     matches 141, differences 2
ENSG000000

In [155]:
# Results are very different between old cr4 version and new filtered version
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_orig[condition]), len(control_cells_ids_filtered[condition]))
    assert len(control_cells_ids_cr4_orig[condition]) == len(control_cells_ids_filtered[condition])
    
    matches = 0
    for pos, (val1, val2) in enumerate(zip(control_cells_ids_cr4_orig[condition], control_cells_ids_filtered[condition].values())):
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {len(control_cells_ids_cr4_orig[condition]) - matches}")

ENSG00000001497 83 83
     matches 1, differences 82
ENSG00000003509 18 18
     matches 0, differences 18
ENSG00000004779 160 160
     matches 0, differences 160
ENSG00000004897 366 366
     matches 4, differences 362
ENSG00000005007 131 131
     matches 0, differences 131
ENSG00000005100 140 140
     matches 1, differences 139
ENSG00000005175 18 18
     matches 1, differences 17
ENSG00000005194 151 151
     matches 1, differences 150
ENSG00000005448 208 208
     matches 1, differences 207
ENSG00000006634 131 131
     matches 1, differences 130
ENSG00000006695 94 94
     matches 0, differences 94
ENSG00000006712 238 238
     matches 1, differences 237
ENSG00000006715 332 332
     matches 1, differences 331
ENSG00000006744 85 85
     matches 1, differences 84
ENSG00000007168 126 126
     matches 1, differences 125
ENSG00000007866 267 267
     matches 1, differences 266
ENSG00000007923 46 46
     matches 1, differences 45
ENSG00000008018 143 143
     matches 1, differences 142
ENSG000000

In [23]:
# But the differences appear to be mostly because of sort order. There are still some unexplained issues with the original dictionary because differences of 1 can't be explained by sorting
for condition in conditions:
    print(condition, len(control_cells_ids_cr4_validate[condition]), len(control_cells_ids_filtered[condition]))
    assert len(control_cells_ids_cr4_validate[condition]) == len(control_cells_ids_filtered[condition])
    
    matches = 0
    num_pairs = len(control_cells_ids_cr4_validate[condition])
    for pos, key in enumerate(control_cells_ids_cr4_validate[condition].keys()):
        val1, val2 = control_cells_ids_cr4_validate[condition][key], control_cells_ids_filtered[condition][key]
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {num_pairs - matches}")

ENSG00000001497 83 83
     matches 83, differences 0
ENSG00000003509 18 18
     matches 18, differences 0
ENSG00000004779 160 160
     matches 160, differences 0
ENSG00000004897 366 366
     matches 366, differences 0
ENSG00000005007 131 131
     matches 131, differences 0
ENSG00000005100 140 140
     matches 140, differences 0
ENSG00000005175 18 18
     matches 18, differences 0
ENSG00000005194 151 151
     matches 151, differences 0
ENSG00000005448 208 208
     matches 208, differences 0
ENSG00000006634 131 131
     matches 131, differences 0
ENSG00000006695 94 94
     matches 94, differences 0
ENSG00000006712 238 238
     matches 238, differences 0
ENSG00000006715 332 332
     matches 332, differences 0
ENSG00000006744 85 85
     matches 85, differences 0
ENSG00000007168 126 126
     matches 126, differences 0
ENSG00000007866 267 267
     matches 267, differences 0
ENSG00000007923 46 46
     matches 46, differences 0
ENSG00000008018 143 143
     matches 143, differences 0
ENSG000000

## Unused Cells

In [13]:
control_cells_used = set()
for condition in conditions:
    control_cells_used = control_cells_used.union(set(control_cells_ids_filtered[condition].keys()))
    control_cells_used = control_cells_used.union(set(control_cells_ids_filtered[condition].values()))
    

In [14]:
all_cells = set(adata_filtered.obs_names)

In [15]:
len(all_cells), len(control_cells_used)

(310385, 310281)

In [16]:
control_cells_used.issubset(all_cells)

True

In [17]:
unused_cell_ids = all_cells.difference(control_cells_used)

In [18]:
unused_cells = adata_filtered.obs.loc[list(unused_cell_ids)]

In [19]:
unused_cells.condition.astype(str).unique()

array(['non-targeting'], dtype=object)

In [20]:
unused_cells

,gem_group,condition_name,condition,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count
cell_barcode,,,,,,,,,,,
TCCACGTGTCATCACA-26,26,non-targeting,non-targeting,non-targeting,11179_non-targeting_non-targeting_non-targeting,non-targeting_02787|non-targeting_01465,0.098578,18777.0,1.571038,0.901381,20831.373047
GGGACTCTCACCTCTG-25,25,non-targeting,non-targeting,non-targeting,10772_non-targeting_non-targeting_non-targeting,non-targeting_00166|non-targeting_02207,0.119748,12877.0,0.203468,0.855594,15050.363281
CCATAAGTCGACGACC-6,6,non-targeting,non-targeting,non-targeting,10822_non-targeting_non-targeting_non-targeting,non-targeting_00442|non-targeting_01312,0.078876,12995.0,-0.127853,0.954127,13619.778320
CCACGAGCATTCTCTA-19,19,non-targeting,non-targeting,non-targeting,10884_non-targeting_non-targeting_non-targeting,non-targeting_00807|non-targeting_02911,0.080580,17523.0,0.372891,1.123384,15598.401367
TTGAACGAGAGTATAC-8,8,non-targeting,non-targeting,non-targeting,10859_non-targeting_non-targeting_non-targeting,non-targeting_00652|non-targeting_02540,0.086616,18634.0,1.354366,0.950474,19604.958984
...,...,...,...,...,...,...,...,...,...,...,...
AATTTCCTCCGAAGGA-12,12,non-targeting,non-targeting,non-targeting,10882_non-targeting_non-targeting_non-targeting,non-targeting_00799|non-targeting_00833,0.093719,13978.0,-0.502728,1.232812,11338.302734
TACCGGGCATGTTCGA-34,34,non-targeting,non-targeting,non-targeting,10872_non-targeting_non-targeting_non-targeting,non-targeting_00702|non-targeting_02833,0.106195,15029.0,-0.047092,1.091537,13768.660156
CATCGTCTCGTTACCC-12,12,non-targeting,non-targeting,non-targeting,11023_non-targeting_non-targeting_non-targeting,non-targeting_01798|non-targeting_01119,0.114911,15525.0,-0.266281,1.232812,12593.157227


## Prototype Algorithm

In [147]:
from collections import defaultdict
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment

precision = 3
np.set_printoptions(precision=precision)

# # PREPROCESSING: Calculate number of non-zero genes
# n_nonzero_genes = (adata_filtered.X > 0).sum(axis=1)
# if not isinstance(n_nonzero_genes, np.ndarray):
#     n_nonzero_genes = n_nonzero_genes.toarray()

# adata_filtered.obs['n_nonzero_genes'] = n_nonzero_genes
# assert (adata_filtered.obs['UMI_count'] >= adata_filtered.obs['n_nonzero_genes']).all()

# Function inputs -- and obs
condition_key = 'condition'
group_key = 'gem_group'
dist_keys = ['UMI_count', 'n_nonzero_genes'] 
control_condition = "non-targeting"

def get_matched_controls_2(obs_metadata: pd.Dataframe, 
                           dist_keys: List[str], 
                           condition_key: str = 'condition', 
                           control_condition: str = 'non-targeting', 
                           group_key='gem_group') -> List[Dict[dict]]:

    
    df_columns = [condition_key, group_key] + dist_keys
    obs_metadata = obs_metadata[df_columns]
    
    control_mask = obs_metadata[condition_key] == control_condition
    control_cells = obs_metadata.loc[control_mask]
    treatment_cells = obs_metadata.loc[~control_mask]
    
    control_cells_gem = {gem_group:df for gem_group,df in control_cells.groupby(group_key)}
    control_cells_ids_filtered_lsum = defaultdict(dict)
    control_cells_ids_filtered_argm = defaultdict(dict)
    
    for condition, treatment_cells_cond in treatment_cells.groupby(condition_key):
        for gem_group, treatment_cells_cond_group in treatment_cells_cond.groupby(group_key):
            control_ = control_cells_gem[gem_group][dist_keys]
            treatment_ = treatment_cells_cond_group[dist_keys]
            if len(treatment_) > len(control_):
                print(f"Warning: {condition}, {gem_group} num treatment cells exceeds num control cells. {len(treatment_)} > {len(control_)}")
            elif len(treatment_) == len(control_):
                print(f"Info: {condition}, {gem_group} num treatment cells equals num control cells. {len(treatment_)} = {len(control_)}")
            
            dist_matrix = pairwise_distances(treatment_, control_)
            treatment_indices, control_indices = linear_sum_assignment(dist_matrix)
            if max(treatment_indices) >= len(treatment_): 
                AssertionError("Treatment indices out of range")
            if max(control_indices) >= len(control_): 
                AssertionError("Control indices out of range")

            control_indices_arg = dist_matrix.argmin(axis=1)
    
            if not np.allclose(control_indices[treatment_indices], control_indices_arg):
                lsum_dist_values = dist_matrix[treatment_indices, control_indices]
                # arg1_dist_values = dist_matrix[treatment_indices, control_indices_arg]
                arg1_dist_values = dist_matrix[list(range(dist_matrix.shape[0])), control_indices_arg]
                print(
                    f'Condition {condition}, gem_group {gem_group} :\n'
                    f'     lsum indices {control_indices[treatment_indices]} distances {lsum_dist_values} {sum(lsum_dist_values):.{precision}f}\n'
                    f'     arg1 indices {control_indices_arg} distances {arg1_dist_values} {sum(arg1_dist_values):.{precision}f}\n'
                )
            
            treatment_ids = treatment_.index[treatment_indices]
            control_ids = control_.index[control_indices]
            control_ids_arg = control_.index[control_indices_arg]
            control_cells_ids_filtered_lsum[condition].update(dict(zip(treatment_ids, control_ids)))
            control_cells_ids_filtered_argm[condition].update(dict(zip(treatment_.index, control_ids_arg)))
    


Condition ENSG00000001497, gem_group 39 :
     lsum indices [156 101 106  12 138] distances [13.928 75.425 39.013 84.876 92.849] 306.092
     arg1 indices [156 101 106  12 156] distances [13.928 75.425 39.013 84.876 82.292] 295.535

Condition ENSG00000004779, gem_group 9 :
     lsum indices [188  32 104  63] distances [106.85  257.391  65.803 155.544] 585.588
     arg1 indices [188  32 104 104] distances [106.85  257.391  65.803  26.571] 456.615

Condition ENSG00000004779, gem_group 33 :
     lsum indices [169 221  13  67  92 238  41 249  60] distances [128.413  48.836 110.476 103.586 168.814  82.735  64.815  35.057  30.61 ] 773.343
     arg1 indices [ 92 221  13  67  92 238  41 249  60] distances [ 82.492  48.836 110.476 103.586 168.814  82.735  64.815  35.057  30.61 ] 727.422

Condition ENSG00000004897, gem_group 12 :
     lsum indices [ 84  20 183 192  78  40 163  70] distances [  7.071  49.98   42.012  72.028 299.868 203.226 112.579  83.522] 870.287
     arg1 indices [ 84  20 183 1

In [127]:
# # Can probably accelerate by inverting loop order
# # Will run fewer pairwise distance calculations, albeit with memory increase
# for gem_group, treatment_gem_group in treatment_cells.groupby(group_key):
#     control_ = control_cells_gem[gem_group][dist_keys]
#     treatment_ = treatment_gem_group[dist_keys]
#     dist_matrix = pairwise_distances(treatment_, control_)
    
#     for condition, treatment_gem_condition in treatment_gem_group.groupby(condition_key):
#         treatment_indices, control_indices = linear_sum_assignment(dist_matrix[treatment_gem_condition.indexes])

#         treatment_ids = treatment_.index[treatment_indices]
#         control_ids = control_.index[control_indices]
#         control_cells_ids[condition].update(dict(zip(treatment_ids, control_ids)))

In [157]:
for condition in conditions:
    print(condition, len(control_cells_ids_filtered[condition]), len(control_cells_ids_filtered_lsum[condition]))
    assert len(control_cells_ids_filtered[condition]) == len(control_cells_ids_filtered_lsum[condition])
    
    matches = 0
    num_pairs = len(control_cells_ids_filtered[condition])
    for pos, key in enumerate(control_cells_ids_filtered[condition].keys()):
        val1, val2 = control_cells_ids_filtered[condition][key], control_cells_ids_filtered_lsum[condition][key]
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {num_pairs - matches}")

ENSG00000001497 83 83
     matches 45, differences 38
ENSG00000003509 18 18
     matches 9, differences 9
ENSG00000004779 160 160
     matches 63, differences 97
ENSG00000004897 366 366
     matches 174, differences 192
ENSG00000005007 131 131
     matches 58, differences 73
ENSG00000005100 140 140
     matches 55, differences 85
ENSG00000005175 18 18
     matches 8, differences 10
ENSG00000005194 151 151
     matches 59, differences 92
ENSG00000005448 208 208
     matches 82, differences 126
ENSG00000006634 131 131
     matches 50, differences 81
ENSG00000006695 94 94
     matches 41, differences 53
ENSG00000006712 238 238
     matches 140, differences 98
ENSG00000006715 332 332
     matches 148, differences 184
ENSG00000006744 85 85
     matches 43, differences 42
ENSG00000007168 126 126
     matches 71, differences 55
ENSG00000007866 267 267
     matches 117, differences 150
ENSG00000007923 46 46
     matches 20, differences 26
ENSG00000008018 143 143
     matches 56, differences 87

In [149]:
for condition in conditions:
    print(condition, len(control_cells_ids_filtered[condition]), len(control_cells_ids_filtered_argm[condition]))
    assert len(control_cells_ids_filtered[condition]) == len(control_cells_ids_filtered_argm[condition])
    
    matches = 0
    num_pairs = len(control_cells_ids_filtered[condition])
    for pos, key in enumerate(control_cells_ids_filtered[condition].keys()):
        val1, val2 = control_cells_ids_filtered[condition][key], control_cells_ids_filtered_argm[condition][key]
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {num_pairs - matches}")

ENSG00000001497 83 83
     matches 46, differences 37
ENSG00000003509 18 18
     matches 9, differences 9
ENSG00000004779 160 160
     matches 64, differences 96
ENSG00000004897 366 366
     matches 175, differences 191
ENSG00000005007 131 131
     matches 58, differences 73
ENSG00000005100 140 140
     matches 55, differences 85
ENSG00000005175 18 18
     matches 8, differences 10
ENSG00000005194 151 151
     matches 59, differences 92
ENSG00000005448 208 208
     matches 83, differences 125
ENSG00000006634 131 131
     matches 51, differences 80
ENSG00000006695 94 94
     matches 41, differences 53
ENSG00000006712 238 238
     matches 120, differences 118
ENSG00000006715 332 332
     matches 147, differences 185
ENSG00000006744 85 85
     matches 42, differences 43
ENSG00000007168 126 126
     matches 71, differences 55
ENSG00000007866 267 267
     matches 118, differences 149
ENSG00000007923 46 46
     matches 20, differences 26
ENSG00000008018 143 143
     matches 58, differences 8

In [150]:
for condition in conditions:
    print(condition, len(control_cells_ids_filtered_lsum[condition]), len(control_cells_ids_filtered_argm[condition]))
    assert len(control_cells_ids_filtered_lsum[condition]) == len(control_cells_ids_filtered_argm[condition])
    
    matches = 0
    num_pairs = len(control_cells_ids_filtered_lsum[condition])
    for pos, key in enumerate(control_cells_ids_filtered_lsum[condition].keys()):
        val1, val2 = control_cells_ids_filtered_lsum[condition][key], control_cells_ids_filtered_argm[condition][key]
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {num_pairs - matches}")

ENSG00000001497 83 83
     matches 82, differences 1
ENSG00000003509 18 18
     matches 18, differences 0
ENSG00000004779 160 160
     matches 158, differences 2
ENSG00000004897 366 366
     matches 356, differences 10
ENSG00000005007 131 131
     matches 130, differences 1
ENSG00000005100 140 140
     matches 134, differences 6
ENSG00000005175 18 18
     matches 18, differences 0
ENSG00000005194 151 151
     matches 150, differences 1
ENSG00000005448 208 208
     matches 197, differences 11
ENSG00000006634 131 131
     matches 129, differences 2
ENSG00000006695 94 94
     matches 93, differences 1
ENSG00000006712 238 238
     matches 188, differences 50
ENSG00000006715 332 332
     matches 327, differences 5
ENSG00000006744 85 85
     matches 84, differences 1
ENSG00000007168 126 126
     matches 125, differences 1
ENSG00000007866 267 267
     matches 263, differences 4
ENSG00000007923 46 46
     matches 46, differences 0
ENSG00000008018 143 143
     matches 140, differences 3
ENSG000

In [168]:
import matplotlib.pyplot as plt

control_cells_ids_filtered_df = pd.DataFrame(pd.DataFrame(control_cells_ids_filtered).stack())
control_cells_ids_filtered_argm_df = pd.DataFrame(pd.DataFrame(control_cells_ids_filtered_argm).stack())
control_cells_ids_filtered_lsum_df = pd.DataFrame(pd.DataFrame(control_cells_ids_filtered_lsum).stack())

In [163]:
control_cells_ids_filtered_df.head()

AAACCCAAGAAATCCA-27  ENSG00000145414    GAGGGATTCGTGGACC-27
AAACGAAAGTCATCGT-17  ENSG00000145414    AGTGCCGGTCATGACT-17
AATGCCACAGCTGTAT-47  ENSG00000145414    TGCGGCAAGACAACTA-47
ACAGAAATCAACTGAC-10  ENSG00000145414    AAATGGACAGAGTAAT-10
ACATCCCTCTGACGCG-34  ENSG00000145414    CAACCTCAGGCACTAG-34
dtype: object

In [166]:
pd.DataFrame(control_cells_ids_filtered_df).head()

,,0
AAACCCAAGAAATCCA-27,ENSG00000145414,GAGGGATTCGTGGACC-27
AAACGAAAGTCATCGT-17,ENSG00000145414,AGTGCCGGTCATGACT-17
AATGCCACAGCTGTAT-47,ENSG00000145414,TGCGGCAAGACAACTA-47
ACAGAAATCAACTGAC-10,ENSG00000145414,AAATGGACAGAGTAAT-10
ACATCCCTCTGACGCG-34,ENSG00000145414,CAACCTCAGGCACTAG-34
